Cell 1

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

Cell 2

In [ ]:
feature_df = pd.read_parquet("data/processed/feature_df.parquet")

groups = feature_df["user"].astype(str)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(feature_df, feature_df["is_insider"], groups=groups))

train_df = feature_df.iloc[train_idx].copy()
test_df = feature_df.iloc[test_idx].copy()

print("feature_df:", feature_df.shape)
print("train_df:", train_df.shape, "test_df:", test_df.shape)
print("train positives:", int(train_df["is_insider"].sum()), "test positives:", int(test_df["is_insider"].sum()))
print("user overlap:", len(set(train_df["user"]).intersection(set(test_df["user"]))))

Cell 3

In [ ]:
leak_cols = [
    "after_hours_events",
    "user_mean_after",
    "user_std_after",
    "after_hours_events_dev",
    "after_hours_ratio",
]
mean_std_cols = [c for c in feature_df.columns if c.startswith("user_mean_") or c.startswith("user_std_")]
drop_cols = list(set(leak_cols + mean_std_cols + ["user", "day", "employee_name", "email", "projects", "total_events"]))


train_clean = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns]).copy()
test_clean = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns]).copy()

y_train = train_clean.pop("is_insider").astype(int)
y_test = test_clean.pop("is_insider").astype(int)

cat_cols = [c for c in ["role", "functional_unit", "department", "team", "supervisor"] if c in train_clean.columns]
X_train = pd.get_dummies(train_clean, columns=cat_cols, drop_first=False)
X_test = pd.get_dummies(test_clean, columns=cat_cols, drop_first=False)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

Cell 4: Logistic Regression

In [ ]:
lr = LogisticRegression(class_weight="balanced", max_iter=5000, n_jobs=-1)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

lr_results = {
    "model": "Logistic Regression",
    "tn": confusion_matrix(y_test, lr_pred)[0, 0],
    "fp": confusion_matrix(y_test, lr_pred)[0, 1],
    "fn": confusion_matrix(y_test, lr_pred)[1, 0],
    "tp": confusion_matrix(y_test, lr_pred)[1, 1],
    "precision": classification_report(y_test, lr_pred, output_dict=True, digits=4)["1"]["precision"],
    "recall": classification_report(y_test, lr_pred, output_dict=True, digits=4)["1"]["recall"],
    "f1": classification_report(y_test, lr_pred, output_dict=True, digits=4)["1"]["f1-score"],
    "roc_auc": roc_auc_score(y_test, lr_proba),
    "pr_auc": average_precision_score(y_test, lr_proba),
}
print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred, digits=4))
print("ROC-AUC:", lr_results["roc_auc"])
print("PR-AUC:", lr_results["pr_auc"])

Cell 5: Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_results = {
    "model": "Random Forest",
    "tn": confusion_matrix(y_test, rf_pred)[0, 0],
    "fp": confusion_matrix(y_test, rf_pred)[0, 1],
    "fn": confusion_matrix(y_test, rf_pred)[1, 0],
    "tp": confusion_matrix(y_test, rf_pred)[1, 1],
    "precision": classification_report(y_test, rf_pred, output_dict=True, digits=4)["1"]["precision"],
    "recall": classification_report(y_test, rf_pred, output_dict=True, digits=4)["1"]["recall"],
    "f1": classification_report(y_test, rf_pred, output_dict=True, digits=4)["1"]["f1-score"],
    "roc_auc": roc_auc_score(y_test, rf_proba),
    "pr_auc": average_precision_score(y_test, rf_proba),
}
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred, digits=4))
print("ROC-AUC:", rf_results["roc_auc"])
print("PR-AUC:", rf_results["pr_auc"])

Cell 6: XGBoost

In [ ]:
spw_values = [1, 10, 50, 100, 300]
xgb_trials = []

for spw in spw_values:
    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        min_child_weight=1,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        scale_pos_weight=spw
    )
    xgb.fit(X_train, y_train)
    proba = xgb.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rep = classification_report(y_test, pred, output_dict=True, digits=4)
    xgb_trials.append({
        "spw": spw,
        "tn": confusion_matrix(y_test, pred)[0, 0],
        "fp": confusion_matrix(y_test, pred)[0, 1],
        "fn": confusion_matrix(y_test, pred)[1, 0],
        "tp": confusion_matrix(y_test, pred)[1, 1],
        "precision": rep["1"]["precision"],
        "recall": rep["1"]["recall"],
        "f1": rep["1"]["f1-score"],
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
        "xgb_pred": xgb.predict(X_test),
        "xgb_model": xgb
    })

xgb_results_df = pd.DataFrame(xgb_trials).sort_values(["f1", "pr_auc"], ascending=False)
xgb_results_df

Cell 7: Final Comparison

### scale_pos_weight (handling class imbalance)

`scale_pos_weight` is a parameter in models like XGBoost that helps deal with imbalanced datasets (when one class is much rarer than the other).

- It increases the importance of positive class samples during training.
- Internally, it scales the gradients (errors) for positive examples, making the model focus more on them.
- This helps improve performance on the minority class (e.g., better recall).

In [ ]:
best_xgb = xgb_results_df.iloc[0].to_dict()
best_xgb["model"] = f"XGBoost (spw={best_xgb['spw']})"
best_xgb = {k: v for k, v in best_xgb.items() if k != "spw"}

results_df = pd.DataFrame([lr_results, rf_results, best_xgb])
results_df = results_df[["model", "tn", "fp", "fn", "tp", "precision", "recall", "f1", "roc_auc", "pr_auc"]]
results_df.sort_values("f1", ascending=False)

Cell 8: Save outputs

In [ ]:
results_df.to_csv("data/processed/binary_model_comparison.csv", index=False)
xgb_results_df.to_csv("data/processed/xgb_spw_trials.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

xgb_pred = best_xgb['xgb_pred']

models = {
    "Logistic Regression": lr_pred,
    "Random Forest": rf_pred,
    "XGBoost": xgb_pred
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, pred) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()
plt.savefig('data/visualizations/LR_RF_XGB_Confusion_Matrix.png', dpi=300, bbox_inches='tight')

In [ ]:
from xgboost import plot_importance

plt.figure(figsize=(10, 6))
plot_importance(best_xgb["xgb_model"], max_num_features=15, importance_type="gain")
plt.tight_layout()
plt.show()
plt.savefig('data/visualizations/XGB_plot_importance.png', dpi=300, bbox_inches='tight')

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

processed = Path('data/processed')

for name in ['train_df', 'test_df']:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        df = globals()[name]
        df.to_csv(processed / f'{name}.csv', index=False)
        df.to_parquet(processed / f'{name}.parquet', index=False)

print('Saved files to data/processed and data/visualizations')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score

xgb_proba = best_xgb["xgb_model"].predict_proba(X_test)[:, 1]

# use your XGBoost probabilities from the holdout split
probs = xgb_proba if "xgb_proba" in globals() else best_xgb["xgb_model"].predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, probs)

f1_scores = []
for t in thresholds:
    pred = (probs >= t).astype(int)
    f1_scores.append(f1_score(y_test, pred, zero_division=0))

f1_scores = np.array(f1_scores)
best_idx = int(np.argmax(f1_scores))
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

best_pred = (probs >= best_threshold).astype(int)

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)
print("Precision at best threshold:", precision_score(y_test, best_pred, zero_division=0))
print("Recall at best threshold:", recall_score(y_test, best_pred, zero_division=0))
print("Confusion matrix at best threshold:")
print(confusion_matrix(y_test, best_pred))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precision[:-1], label="Precision")
plt.plot(thresholds, recall[:-1], label="Recall")
plt.plot(thresholds, f1_scores, label="F1")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best threshold = {best_threshold:.3f}")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("XGBoost threshold tuning")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import joblib

joblib.dump(best_xgb['xgb_model'], "data/models/xgb_final.joblib")
print("Saved to models/xgb_final.joblib")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

xgb_pred = best_xgb['xgb_pred']

models = {
    "Logistic Regression": (lr_pred, "Blues"),
    "Random Forest": (rf_pred, "Greens"),
    "XGBoost": (xgb_pred, "Oranges")
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

labels = np.array([["TN", "FP"],
                   ["FN", "TP"]])

for ax, (name, (pred, cmap)) in zip(axes, models.items()):
    
    cm = confusion_matrix(y_test, pred)

    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)

    # Tick labels
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Negative', 'Positive'], fontsize=11)
    ax.set_yticklabels(['Negative', 'Positive'], fontsize=11)

    ax.set_xlabel("Predicted Label", fontsize=12, fontweight='bold')
    ax.set_ylabel("True Label", fontsize=12, fontweight='bold')
    ax.set_title(name, fontsize=14, fontweight='bold')

    # Write TN, FP, FN, TP with count
    thresh = cm.max() / 2

    for i in range(2):
        for j in range(2):
            color = "white" if cm[i, j] > thresh else "black"
            ax.text(
                j, i,
                f"{labels[i, j]}\n{cm[i, j]}",
                ha="center",
                va="center",
                fontsize=13,
                fontweight="bold",
                color=color
            )

plt.tight_layout()

plt.savefig(
    'data/visualizations/LR_RF_XGB_Confusion_Matrix.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()